In [92]:
import csv

filename = 'NS2_databaseFinished.csv'
keys =['summary', 'coordinates', 'iyear', 'imonth', 'iday', 'gname', 'weaptype1_txt',
        'location', 'targsubtype1_txt', 'target1', 'success']
records = []

with open(filename, 'r') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        records.append({key: row[key] for key in keys})

In [93]:
print(records[0])

{'summary': '01/12/2005: Unknown perpetrators threw a grenade into a cinema in Mogadishu, Somalia. The explosion and resulting stampede killed three people and left nine others injured.', 'coordinates': 'POINT (2.059819, 45.326115)', 'iyear': '2005', 'imonth': '1', 'iday': '12', 'gname': 'Unknown', 'weaptype1_txt': 'Explosives', 'location': '', 'targsubtype1_txt': 'Entertainment/Cultural/Stadium/Casino', 'target1': 'Civilians in cinema hall', 'success': '1'}


In [94]:
for record in records:
    if record['coordinates']:
        latitude, longitude = record['coordinates'].split("(")[-1].split(")")[0].replace(",", "").split()
        record['latitude'] = float(latitude)
        record['longitude'] = float(longitude)
    else: 
        record['latitude'] = None
        record['longitude'] = None


In [96]:
import html
import folium
import ipywidgets as widgets
from folium.plugins import MarkerCluster

years = sorted({int(a['iyear']) for a in records})
groups = ['All', 'Al-Shabaab', 'Islamic Courts Union (ICU)', 'Al-Ittihaad al-Islami (AIAI)']
weapons = ['All', 'Explosives', 'Firearms', 'Unknown', 'Melee', 'Incendiary', 'Other', 'Chemical', 'Sabotage Equipment']

def draw_map(year_range, month, group, weapon):
    m = folium.Map(location=[2.6667, 39.0667], zoom_start=4)
    cluster = MarkerCluster().add_to(m)
    for a in records:
        if (a['latitude'] is not None
                and year_range[0] <= int(a['iyear']) <= year_range[1]
                and (month == 0 or int(a['imonth']) == month)
                and (group == 'All' or a['gname'] == group)
                and (weapon == 'All' or a['weaptype1_txt'] == weapon)):
            popup_html = f"""
            <b>{html.escape(a['gname'])}</b><br>
            <b>Location:</b> {html.escape(a['location'] or 'N/A')}<br>
            <b>Target:</b> {html.escape(a['target1'] or 'N/A')}<br>
            <b>Target type:</b> {html.escape(a['targsubtype1_txt'] or 'N/A')}<br>
            <b>Weapon:</b> {html.escape(a['weaptype1_txt'] or 'N/A')}<br>
            <b>Success:</b> {'Yes' if a['success'] == '1' else 'No'}<br><br>
            {html.escape(a['summary'])}
            """
            folium.Marker(
                [a['latitude'], a['longitude']],
                popup=folium.Popup(popup_html, max_width=350),
            ).add_to(cluster)
    return m

widgets.interact(
    draw_map,
    year_range=widgets.IntRangeSlider(min=years[0], max=years[-1], value=[years[0], years[-1]], continuous_update=False),
    month=widgets.IntSlider(min=0, max=12, value=0),
    group=widgets.Dropdown(options=groups, value='All'),
    weapon=widgets.Dropdown(options=weapons, value='All')
)

interactive(children=(IntRangeSlider(value=(2005, 2015), continuous_update=False, description='year_range', ma…

<function __main__.draw_map(year_range, month, group, weapon)>